<div class="doris-cover">
  <div class="doris-cover-kicker">DEMO 04 · DBT × APACHE DORIS</div>
  <div class="doris-cover-title">Late-Arriving Orders Incremental</div>
  <p class="doris-cover-lead">Simulate a delayed order correction, update the current order version with an incremental merge, and verify idempotency.</p>
  <span class="doris-cover-note">Incremental · merge · Unique Key · on_schema_change · Data Test</span>
</div>

## 1. Check the execution environment

Run this cell first. It uses the Demo dbt environment and current Doris connection settings, then confirms that a Backend is available.

In [ ]:
import importlib.util
from pathlib import Path


def find_demo_dir(start):
    for candidate in (start, *start.parents):
        demo_dir = candidate / "examples/doris-demos"
        if demo_dir.is_dir():
            return demo_dir
    raise FileNotFoundError("Start Jupyter from the dbt-for-apache-doris repository or a subdirectory.")


demo_dir = find_demo_dir(Path.cwd().resolve())
helper_path = demo_dir / "scripts/notebook_helpers.py"
helper_spec = importlib.util.spec_from_file_location("dbt_doris_notebook_helpers", helper_path)
notebook_helpers = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(notebook_helpers)

runner = notebook_helpers.DemoRunner()
runner.show_environment()

## 2. Demo 4: Late-Arriving Orders Incremental

An order system may send a price correction or backfill after the original event has landed. Summing every event can duplicate an order or retain an outdated amount. This Demo uses order versions and incremental merging to maintain the current state of each order.

<table class="doris-index">
  <tr><th>Business users</th><td>Order operations, finance data, and data platform teams</td></tr>
  <tr><th>Business question</th><td>How should late corrections and new orders update current orders and revenue without duplicates?</td></tr>
  <tr><th>Metric rule</th><td><code>order_id</code> is the business key; the latest <code>created_at</code> event is current</td></tr>
  <tr><th>Delivered datasets</th><td>Current order table <code>incremental_daily_sales</code> plus revenue, latency, and quality models</td></tr>
</table>

The cells below show how dbt-doris uses a Unique Key and `merge` to update the current version after source events change.

<div class="doris-flow">
  <div class="doris-flow-step"><strong>Initial events</strong>3 order events</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>Identify versions</strong><code>order_version_history</code></div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>Initial load</strong>Unique Key Incremental Table</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>Delayed update</strong>New version 101 + new order 104</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>Merge result</strong>4 current order versions</div>
</div>

### 2.1 Prepare and inspect initial order events

The source table stores orders as events. Orders 101, 102, and 103 each have one event; a new version of 101 and order 104 arrive later.

In [ ]:
incremental_dir = runner.examples_root / "doris-late-arriving-orders"
runner.show_file("Fixture SQL", incremental_dir / "scripts/setup.sql")
runner.show_file("Source declarations", incremental_dir / "models/sources.yml")
runner.run_sql_file("Create incremental source table", incremental_dir / "scripts/setup.sql")
runner.query("Input: initial order events", """
select event_id, order_id, customer_id, channel_id, grand_total, ordered_at, created_at
from dbt_demo_incremental_source.ORDERS
order by event_id
""")

### 2.2 Build version history and the Incremental table

`order_version_history` ranks events for each order with window functions. `incremental_daily_sales` outputs only `is_current = 1` versions and configures `unique_key='order_id'` with `incremental_strategy='merge'`.

In [ ]:
runner.show_file("Version history model", incremental_dir / "models/order_version_history.sql")
runner.show_file("Incremental Model", incremental_dir / "models/incremental_daily_sales.sql")
runner.show_file("Incremental test definition", incremental_dir / "models/incremental.yml")
runner.run_dbt("Initial full build", incremental_dir, "build")
runner.query("Intermediate result: current order versions", """
select order_id, order_date, grand_total, version_num
from dbt_demo_incremental.incremental_daily_sales
order by order_id
""")

### 2.3 Insert the delayed version and new order

The new events change order 101 from 100.00 to 125.00 and add order 104. The target table is unchanged at this point; only the source event table changes.

In [ ]:
late_events_sql = """
insert into dbt_demo_incremental_source.ORDERS values
    (4, 101, 1, 'web', 125.00, 'COMPLETED', '2026-08-01 09:00:00', '2026-08-05 09:00:00'),
    (5, 104, 3, 'mobile', 70.00, 'COMPLETED', '2026-08-01 12:00:00', '2026-08-05 10:00:00')
"""
runner.show_sql("Delayed event SQL", late_events_sql)
runner.run_sql("Insert delayed version and new order", late_events_sql)
runner.query("Changed source events", """
select event_id, order_id, grand_total, created_at
from dbt_demo_incremental_source.ORDERS
order by event_id
""")

### 2.4 Run the incremental merge

The second `dbt build` recalculates dependent models. The Incremental model merges old and new records by `order_id`: 101 becomes version 2 and 104 becomes version 1.

In [ ]:
runner.run_dbt("Run incremental merge", incremental_dir, "build")
runner.query("Current versions after merge", """
select order_id, order_date, grand_total, version_num
from dbt_demo_incremental.incremental_daily_sales
order by order_id
""")
runner.query("Daily revenue after merge", """
select order_date, order_count, total_revenue
from dbt_demo_incremental.daily_sales_summary
order by order_date
""")

### 2.5 Run again to confirm idempotency

Run the same build again without new source data. The result should remain four orders and 245.00 revenue for August 1.

In [ ]:
runner.run_dbt("Run the Incremental model again", incremental_dir, "build")
runner.run_script("Verify incremental Demo", incremental_dir / "scripts/verify.sh")
runner.query("Result after idempotent run", """
select count(*) as order_rows, count(distinct order_id) as distinct_orders,
       sum(case when order_date = '2026-08-01' then grand_total else 0 end) as aug_01_revenue
from dbt_demo_incremental.incremental_daily_sales
""")

## Complete

The initial build, delayed update, Unique Key merge, and no-change rerun all passed verification.